# ComfyUI ControlNet Batch Generator

GitHub stores the project; Colab provides the GPU. Outputs are saved directly to Google Drive and the generator resumes after disconnects.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Configure

Use a checkpoint and Canny ControlNet from the same model family.


In [ ]:
GITHUB_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
GITHUB_BRANCH = "main"

CHECKPOINT_URL = ""
CHECKPOINT_FILENAME = "YOUR_CHECKPOINT.safetensors"
CONTROLNET_URL = ""
CONTROLNET_FILENAME = "YOUR_CANNY_CONTROLNET.safetensors"

REFERENCE_IMAGE = "/content/drive/MyDrive/comfy-batch/reference.png"
OUTPUT_DIR = "/content/drive/MyDrive/comfy-batch/outputs"
MODEL_CACHE = "/content/drive/MyDrive/comfy-batch/models"

COUNT = 10              # 1 / 10 / 100 / 1000 / 10000
SEED_MODE = "random"    # random / sequential / fixed
BASE_SEED = 123456

COMFYUI_DIR = "/content/ComfyUI"
PROJECT_DIR = "/content/comfy-batch-project"


## Clone ComfyUI + your GitHub repo


In [ ]:
import subprocess, shutil
from pathlib import Path

if not Path(COMFYUI_DIR).exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",COMFYUI_DIR],check=True)

if Path(PROJECT_DIR).exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(["git","clone","--depth","1","--branch",GITHUB_BRANCH,GITHUB_REPO,PROJECT_DIR],check=True)
print("Cloned.")


## Install dependencies


In [ ]:
import subprocess, sys

# Install ComfyUI dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", f"{COMFYUI_DIR}/requirements.txt"
], check=True)

# Install this project's dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", f"{PROJECT_DIR}/requirements.txt"
], check=True)

# OpenPose preprocessing support
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "controlnet-aux"
], check=True)

print("Dependencies ready, including OpenPose.")


## Cache / download models

If the model already exists in Drive, it is reused. Otherwise provide a direct downloadable URL.


In [ ]:
from pathlib import Path
import shutil, urllib.request
cache=Path(MODEL_CACHE); cache.mkdir(parents=True,exist_ok=True)
ckpt_dir=Path(COMFYUI_DIR)/"models"/"checkpoints"; ckpt_dir.mkdir(parents=True,exist_ok=True)
cn_dir=Path(COMFYUI_DIR)/"models"/"controlnet"; cn_dir.mkdir(parents=True,exist_ok=True)

def ensure_model(filename,url,dest_dir):
    cached=cache/filename; dest=dest_dir/filename
    if not cached.exists():
        if not url:
            raise FileNotFoundError(f"Missing {cached} and no download URL supplied")
        print("Downloading",filename)
        urllib.request.urlretrieve(url,cached)
    if not dest.exists():
        print("Copying to local Colab SSD",filename)
        shutil.copy2(cached,dest)

ensure_model(CHECKPOINT_FILENAME,CHECKPOINT_URL,ckpt_dir)
ensure_model(CONTROLNET_FILENAME,CONTROLNET_URL,cn_dir)
print("Models ready.")


## Check reference


In [ ]:
from pathlib import Path
assert Path(REFERENCE_IMAGE).exists(), f"Reference not found: {REFERENCE_IMAGE}"
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
print("Reference:",REFERENCE_IMAGE)
print("Output:",OUTPUT_DIR)


## Start ComfyUI headlessly


In [ ]:
import subprocess, time, urllib.request
log=open('/content/comfyui.log','w')
proc=subprocess.Popen(["python",f"{COMFYUI_DIR}/main.py","--listen","127.0.0.1","--port","8188"],stdout=log,stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/object_info",timeout=2)
        print("ComfyUI ready. PID",proc.pid)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ComfyUI did not start; inspect /content/comfyui.log")


## Validate repository


In [ ]:
subprocess.run(["python",f"{PROJECT_DIR}/scripts/validate_project.py"],cwd=PROJECT_DIR,check=True)


## Generate

Safe to rerun. Existing numbered PNG files are skipped.


In [ ]:
cmd=[
    "python",f"{PROJECT_DIR}/scripts/generate.py",
    "--config",f"{PROJECT_DIR}/config/config.json",
    "--count",str(COUNT),
    "--reference",REFERENCE_IMAGE,
    "--checkpoint",CHECKPOINT_FILENAME,
    "--controlnet",CONTROLNET_FILENAME,
    "--output-dir",OUTPUT_DIR,
    "--seed-mode",SEED_MODE,
    "--base-seed",str(BASE_SEED),
]
print(" ".join(cmd))
subprocess.run(cmd,cwd=PROJECT_DIR,check=True)


## Progress


In [ ]:
from pathlib import Path
import csv
out=Path(OUTPUT_DIR)
print("Completed PNGs:",len(list(out.glob("portrait_*.png"))))
manifest=out/"manifest.csv"
if manifest.exists():
    rows=list(csv.DictReader(manifest.open(encoding="utf-8")))
    print("Manifest rows:",len(rows))
    for row in rows[-5:]: print(row["index"],row["status"],row["filename"])


## Resume

If Colab disconnects, reconnect and rerun the notebook with the same `OUTPUT_DIR` and `COUNT`. Completed numbered images are skipped automatically.
